# 04 - Evaluación final: métricas y matriz de confusión

Este notebook entrena la configuración regularizada usando Train y monitorea Dev. Después usa Test para la evaluación final. No debes ajustar hiperparámetros a partir de las métricas de Test.

## 1. Preparar datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

In [ ]:
(x_all, y_all), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_dev, y_train, y_dev = train_test_split(
    x_all, y_all, test_size=0.20, random_state=SEED, stratify=y_all
)
x_train = x_train.astype("float32") / 255.0
x_dev = x_dev.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Train:", x_train.shape, y_train.shape)
print("Dev:", x_dev.shape, y_dev.shape)
print("Test:", x_test.shape, y_test.shape)
print("Rango de píxeles:", x_train.min(), x_train.max())

## 2. Entrenar la configuración final seleccionada

In [ ]:
x_train_reg, _, y_train_reg, _ = train_test_split(
    x_train, y_train, train_size=12000, random_state=SEED, stratify=y_train
)
keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
modelo_reg = keras.Sequential([
    layers.Input(shape=(28, 28)), layers.Flatten(),
    layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.30),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    layers.Dropout(0.30),
    layers.Dense(10, activation="softmax")
])
modelo_reg.compile(optimizer=keras.optimizers.Adam(0.001),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
parada = callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
hist = modelo_reg.fit(x_train_reg, y_train_reg, validation_data=(x_dev, y_dev),
                      epochs=20, batch_size=64, callbacks=[parada], verbose=1)
print("Épocas ejecutadas:", len(hist.history["loss"]))

## 3. Predicción y métricas globales de Test

In [ ]:
probabilidades = modelo_reg.predict(x_test, verbose=0)
y_pred = np.argmax(probabilidades, axis=1)
accuracy = accuracy_score(y_test, y_pred)
precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
metricas = pd.DataFrame({"metrica": ["Accuracy", "Precision macro", "Recall macro", "F1 macro"],
                         "resultado": [accuracy, precision_macro, recall_macro, f1_macro]})
display(metricas.style.format({"resultado": "{:.4f}"}))
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

## 4. Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=np.arange(10)).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d"
)
ax.set_title("Matriz de confusión - MNIST")
plt.tight_layout(); plt.show()

## 5. Confusión más frecuente y ejemplo real mal clasificado

In [ ]:
cm_errores = cm.copy()
np.fill_diagonal(cm_errores, 0)
real, predicho = np.unravel_index(np.argmax(cm_errores), cm_errores.shape)
cantidad = int(cm_errores[real, predicho])
print(f"Confusión más frecuente: real {real} predicho como {predicho} ({cantidad} veces)")
indices = np.where((y_test == real) & (y_pred == predicho))[0]
if len(indices):
    idx = int(indices[0])
    plt.imshow(x_test[idx], cmap="gray")
    plt.title(f"Real: {y_test[idx]} | Predicción: {y_pred[idx]}")
    plt.axis("off"); plt.show()
    print("Índice de muestra:", idx)

reporte = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
por_clase = pd.DataFrame([
    {"digito": int(k), "precision": v["precision"], "recall": v["recall"],
     "f1": v["f1-score"], "support": v["support"]}
    for k, v in reporte.items() if k.isdigit()
])
display(por_clase.sort_values("f1"))

**Análisis para el PDF:** reporta las cuatro métricas; identifica al menos un dígito con Precision, Recall o F1 inferior al promedio; explica que la diagonal de la matriz son aciertos y fuera de ella están las confusiones; interpreta el par más frecuente y el ejemplo mal clasificado. No uses Test para reentrenar o escoger configuraciones.